# NPFL124 Assignment 1: Language Identification
**Chosen languages:**
* English (`eng`),
* Italian (`ita`),
* Spanish (`spa`)

This notebook implements a character n-gram language model to automatically identify the language of a given text, utilizing "add less than one" smoothing and parameter optimization based on a heldout dataset.

## 1. Data collection
We have downloaded public domain books from [Project Gutenberg](https://www.gutenberg.org) to serve as our corpora. The data is saved locally to ensure reproducibility. We then calculate the size of each raw text file in bytes:

In [1]:
import os

print("Downloading corpora...")
!wget -q https://www.gutenberg.org/files/2701/2701-0.txt -O corpus_eng.txt  # Moby Dick (English)
!wget -q https://www.gutenberg.org/files/2000/2000-0.txt -O corpus_spa.txt  # Don Quijote (Spanish)
!wget -q https://www.gutenberg.org/cache/epub/1012/pg1012.txt -O corpus_ita_1.txt   # La Divina Commedia
!wget -q https://www.gutenberg.org/cache/epub/39097/pg39097.txt -O corpus_ita_2.txt # I Promessi Sposi (Full version)
!cat corpus_ita_1.txt corpus_ita_2.txt > corpus_ita.txt

corpora_files = {
    'eng': 'corpus_eng.txt',
    'ita': 'corpus_ita.txt',
    'spa': 'corpus_spa.txt'
}

data_sizes_bytes = {}

for lang, filename in corpora_files.items():
    size_bytes = os.path.getsize(filename)
    data_sizes_bytes[lang] = size_bytes
    print(f"Language [{lang}] - size in bytes: {size_bytes:,}")

Language [eng] - size in bytes: 1,234,609
Language [ita] - size in bytes: 1,152,373
Language [spa] - size in bytes: 2,226,045


**Note for the Italian corpus**: the assignment requires collecting at least 200 thousand tokens per language. Since tokenizing *La Divina Commedia* alone yielded only about 130 thousand tokens, I concatenated it with the full text of *I Promessi Sposi*. This combination passes the required threshold, ensuring a large enough dataset to accurately train the character n-gram models.

## 2. Tokenization and data splitting
I read the raw text and tokenize it using the `sacremoses` library to verify that I have collected at least 200,000 tokens per language. After tokenization, I split the data for each language into three distinct subsets:
* Training (80%),
* Heldout/Validation (10%),
* and Test (10%).

In [2]:
!pip install sacremoses

In [3]:
from sacremoses import MosesTokenizer

splits = {}
data_sizes_tokens = {}

for lang, filename in corpora_files.items():
    print(f"Tokenizing {lang}...")
    tokenizer = MosesTokenizer(lang=lang)

    with open(filename, 'r', encoding='utf-8') as f:
        text = f.read()

    # Tokenize
    tokens = []
    for line in text.split('\n'):
        if line.strip():
            tokens.extend(tokenizer.tokenize(line))

    total_tokens = len(tokens)
    data_sizes_tokens[lang] = total_tokens
    print(f"Language [{lang}] - total tokens: {total_tokens:,}")

    # Data split
    train_split = int(total_tokens * 0.8)
    heldout_split = train_split + int(total_tokens * 0.1)

    splits[lang] = {
        'train': tokens[:train_split],
        'heldout': tokens[train_split:heldout_split],
        'test': tokens[heldout_split:]
    }

Tokenizing eng...
Language [eng] - total tokens: 259,961
Tokenizing ita...
Language [ita] - total tokens: 243,489
Tokenizing spa...
Language [spa] - total tokens: 453,874


## 3. Out-Of-Vocabulary (OOV) tokens

This section computes the percentage of OOV tokens in the **heldout** and **test** sets to quantify the language sparsity across the different data splits.

In [4]:
oov_percentages = {'heldout': {}, 'test': {}}

for lang in corpora_files.keys():
    train_vocab = set(splits[lang]['train'])

    heldout_tokens = splits[lang]['heldout']
    heldout_len = len(heldout_tokens)
    heldout_oov_count = sum(1 for token in heldout_tokens if token not in train_vocab)

    oov_heldout_pct = (heldout_oov_count / heldout_len * 100) if heldout_len > 0 else 0.0
    oov_percentages['heldout'][lang] = oov_heldout_pct

    test_tokens = splits[lang]['test']
    test_len = len(test_tokens)
    test_oov_count = sum(1 for token in test_tokens if token not in train_vocab)

    oov_test_pct = (test_oov_count / test_len * 100) if test_len > 0 else 0.0
    oov_percentages['test'][lang] = oov_test_pct

    print(f"[{lang}]")
    print(f"  - training vocabulary: {len(train_vocab):,} unique tokens")
    print(f"  - OOV heldout: {oov_heldout_pct:.3f}% ({heldout_oov_count:,} out of {heldout_len:,})")
    print(f"  - OOV test:    {oov_test_pct:.3f}% ({test_oov_count:,} out of {test_len:,})\n")

[eng]
  - training vocabulary: 18,042 unique tokens
  - OOV heldout: 5.593% (1,454 out of 25,996)
  - OOV test:    4.050% (1,053 out of 25,997)

[ita]
  - training vocabulary: 20,671 unique tokens
  - OOV heldout: 9.837% (2,395 out of 24,348)
  - OOV test:    9.655% (2,351 out of 24,350)

[spa]
  - training vocabulary: 21,887 unique tokens
  - OOV heldout: 3.728% (1,692 out of 45,387)
  - OOV test:    7.440% (3,377 out of 45,388)



## 5. Character n-gram models (MLE estimation)

This section extracts character unigrams, bigrams, and trigrams from the training data. These raw counts form the basis for our Markov model estimations.

1. Text Reconstruction: the training set tokens are joined with spaces to preserve word boundaries.
2. N-gram Extraction: we compute exact frequencies for $1$-grams, $2$-grams, and $3$-grams using MLE.
3. Vocabulary ($|V|$): we extract the unique character vocabulary size, which is strictly required for the upcoming smoothing phase.

In [5]:
from collections import Counter

unigram_counts = {}
bigram_counts = {}
trigram_counts = {}
vocab_char = {}

for lang in corpora_files.keys():
    train_text = " ".join(splits[lang]['train'])

    # Unigrams
    unigrams = list(train_text)
    unigram_counts[lang] = Counter(unigrams)
    vocab_char[lang] = set(unigrams)

    # Bigrams
    bigrams = [train_text[i:i + 2] for i in range(len(train_text) - 1)]
    bigram_counts[lang] = Counter(bigrams)

    # Trigrams
    trigrams = [train_text[i:i + 3] for i in range(len(train_text) - 2)]
    trigram_counts[lang] = Counter(trigrams)

    total_trigrams = sum(trigram_counts[lang].values())

    print(f"\n[{lang}] - character vocabulary size (|V|): {len(vocab_char[lang])}")
    print(f"Top 5 Character trigrams (total extracted trigrams: {total_trigrams:,}):")

    top_5 = trigram_counts[lang].most_common(5)
    for trigram, count in top_5:
        rel_freq = count / total_trigrams
        print(f"  {repr(trigram)}: count = {count:,} | relative frequency = {rel_freq:.6f}")


[eng] - character vocabulary size (|V|): 98
Top 5 Character trigrams (total extracted trigrams: 1,015,888):
  ' th': count = 19,700 | relative frequency = 0.019392
  ' , ': count = 15,258 | relative frequency = 0.015019
  'the': count = 15,111 | relative frequency = 0.014875
  'he ': count = 13,088 | relative frequency = 0.012883
  'ed ': count = 6,727 | relative frequency = 0.006622

[ita] - character vocabulary size (|V|): 106
Top 5 Character trigrams (total extracted trigrams: 891,367):
  ' , ': count = 13,319 | relative frequency = 0.014942
  'he ': count = 8,850 | relative frequency = 0.009929
  ' ’ ': count = 7,630 | relative frequency = 0.008560
  ' . ': count = 5,996 | relative frequency = 0.006727
  ' ch': count = 5,497 | relative frequency = 0.006167

[spa] - character vocabulary size (|V|): 96
Top 5 Character trigrams (total extracted trigrams: 1,752,601):
  ' , ': count = 32,269 | relative frequency = 0.018412
  ' de': count = 22,603 | relative frequency = 0.012897
  'que'

The extracted trigrams and vocabulary sizes provide a statistically sound foundation for the subsequent modeling phases:

* **Linguistic consistency**: the top trigrams align with the orthographic characteristics of each language. For instance, the dominance of `' th'` and `'the'` in English reflects the high frequency of the definite article, while `' de'` and `'que'` in Spanish represent essential prepositions and conjunctions.
* **Vocabulary and sparsity**: the vocabulary size ($|V| \approx 100$) is consistent across all three languages, accounting for alphanumeric characters, punctuation, and language-specific accents.
* **Data robustness**: with total trigram counts ranging from ~0.9M to ~1.7M, the sample size is sufficient to move toward $\delta$-smoothing without encountering excessive variance in the MLE estimates.

## 6. $\delta$-Smoothing

This section addresses the zero-probability problem using the **add-lambda** ($\delta$-smoothing) technique. We perform a grid search to find the optimal $\lambda$ that minimizes cross-entropy.

1. **Smoothing formula**: we apply
    $$p'(w_i|w_{i-2}, w_{i-1}) = \frac{c_3(w_{i-2}, w_{i-1}, w_i) + \lambda}{c_2(w_{i-2}, w_{i-1}) + \lambda |V|}$$
   to redistribute probability mass to unseen trigrams.
2. **Optimization**: the hyperparameter $\lambda$ is tuned on the **heldout set** to ensure the model generalizes well to unseen data.
3. **Evaluation:** we calculate the cross-entropy ($H$) on the **test set** using both the initial $\lambda$ (0.1) and the optimal $\lambda$ found during the grid search.

In [6]:
import math


def get_smoothed_prob(trigram, history, lang, lambda_param):
    """Calculates smoothed probability p'(w_i|w_{i-2}, w_{i-1}) using add-lambda smoothing."""
    c_trigram = trigram_counts[lang].get(trigram, 0)
    c_history = bigram_counts[lang].get(history, 0)
    v_size = len(vocab_char[lang])

    return (c_trigram + lambda_param) / (c_history + lambda_param * v_size)


def compute_cross_entropy(text_tokens, lang, lambda_param):
    """Calculates the cross-entropy of a tokenized sequence."""
    text = " ".join(text_tokens)
    log_prob_sum = 0.0

    N = len(text) - 2  # N is the number of trigrams (sequence length minus the initial history)

    if N <= 0:
        return float('inf')

    for i in range(2, len(text)):
        history = text[i - 2:i]
        trigram = text[i - 2:i + 1]

        p = get_smoothed_prob(trigram, history, lang, lambda_param)
        log_prob_sum += math.log2(p)

    return -log_prob_sum / N


# Initial configuration
initial_lambda = 0.1
lambdas_grid = [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 0.8, 0.99]
optimal_lambdas = {}

print("Smoothing evaluation and parameter optimization..")

for lang in corpora_files.keys():
    # Initial evaluation on the test set
    initial_test_ce = compute_cross_entropy(splits[lang]['test'], lang, initial_lambda)

    # Grid search on heldout set for optimization
    best_lambda = initial_lambda
    best_heldout_ce = float('inf')

    for l_param in lambdas_grid:
        heldout_ce = compute_cross_entropy(splits[lang]['heldout'], lang, l_param)
        if heldout_ce < best_heldout_ce:
            best_heldout_ce = heldout_ce
            best_lambda = l_param

    optimal_lambdas[lang] = best_lambda

    # Final evaluation on the test set
    final_test_ce = compute_cross_entropy(splits[lang]['test'], lang, best_lambda)

    print(f"\n[{lang}]")
    print(f"\t- Initial smoothing parameter : {initial_lambda}")
    print(f"\t- Initial test cross-entropy  : {initial_test_ce:.4f}")
    print(f"\t- Best smoothing parameter    : {best_lambda}")
    print(f"\t- Final test cross-entropy    : {final_test_ce:.4f}")

Smoothing evaluation and parameter optimization..

[eng]
	- Initial smoothing parameter : 0.1
	- Initial test cross-entropy  : 2.7545
	- Best smoothing parameter    : 0.01
	- Final test cross-entropy    : 2.7495

[ita]
	- Initial smoothing parameter : 0.1
	- Initial test cross-entropy  : 3.1086
	- Best smoothing parameter    : 0.05
	- Final test cross-entropy    : 3.1018

[spa]
	- Initial smoothing parameter : 0.1
	- Initial test cross-entropy  : 2.8781
	- Best smoothing parameter    : 0.01
	- Final test cross-entropy    : 2.8936


The grid search identified optimal $\lambda$ values for all languages.

* For English and Italian, the transition from $\lambda=0.1$ to the optimized parameters resulted in a decrease in cross-entropy, indicating better model generalization.

* In the case of Spanish, the slight increase in cross-entropy on the test set (despite optimization on heldout) highlights the subtle variance between data splits.

* The final cross-entropy values (ranging from ~2.75 to ~3.10) demonstrate that the character-level trigram models have captured the statistical regularities of the three languages effectively.

## 7. Language identification classifier

This final step implements the inference function. To avoid arithmetic underflow caused by multiplying many small probabilities, we perform calculations in **log-space** ($\log_2$).

1.  The input string is prefixed with spaces to provide the necessary history for the initial characters.
2.  Instead of products, we sum the log-probabilities of all trigrams using the optimized $\lambda$ for each language.
3.  The language with the highest (least negative) log-probability is selected as the prediction.

In [8]:
def identify_language(text):
    """
    Identifies the language by comparing log-probabilities across models.
    Returns a sorted list of (log-probability, language) tuples.
    """
    padded_text = "  " + text
    results = []

    for lang in corpora_files.keys():
        log_prob_sum = 0.0
        best_lambda = optimal_lambdas[lang]

        for i in range(2, len(padded_text)):
            history = padded_text[i - 2:i]
            trigram = padded_text[i - 2:i + 1]

            p = get_smoothed_prob(trigram, history, lang, best_lambda)
            log_prob_sum += math.log2(p)

        results.append((log_prob_sum, lang))

    results.sort(key=lambda x: x[0], reverse=True)
    return results


random_sentence = "Nel mezzo del cammin di nostra vita mi ritrovai per una selva oscura."

print(f"Test sentence: '{random_sentence}'\n")
predictions = identify_language(random_sentence)

print("Log-Probability Ranking:")
for prob, lang in predictions:
    print(f"  [{lang}]: {prob:.4f}")

# The model with the highest probability determines the predicted language
predicted_lang = predictions[0][1]
print(f"\nSystem Prediction: {predicted_lang}")

Test sentence: 'Nel mezzo del cammin di nostra vita mi ritrovai per una selva oscura.'

Log-Probability Ranking:
  [ita]: -216.8591
  [spa]: -306.4247
  [eng]: -374.2447

System Prediction: ita
